# CRA Tutorial — Hands-on AI on the National Research Platform

This is the hands-on portion of the tutorial. Everything runs **inside this
JupyterHub session** — no pods to launch, no YAML, and **no GPU required**.

A one-line vocabulary primer:

- **LLM** — a large language model (answers prompts).
- **Inference** — running a prompt through a model to get an answer.
- **RAG** — *retrieval-augmented generation*: look up relevant text first, then
  answer **using only that text**.

**What you'll do**

| Step | Section |
|---|---|
| ① Check the session (env vars, `kubectl`) | 1 |
| ② Call NRP's **managed LLM** — list models, chat, swap models | 2 |
| ③ Build a **RAG** pipeline over NRP's docs with managed embeddings | 3 |

Then [`3_jupyterhub.ipynb`](3_jupyterhub.ipynb) has you deploy your *own*
JupyterHub with Helm.

**Running cells:** grey cells are code — click one and press **Shift + Enter**,
top to bottom. `[*]` = running, `[7]` = done. This notebook uses the **Python 3**
kernel (shown top-right).

**Prerequisites:** a **CPU-only** session is plenty (the 1 core / 8 GB defaults
are fine). The session already exports `OPENAI_API_BASE` / `OPENAI_API_KEY` and
has `kubectl` wired to `nrp-training-k8s`.

## 1. Setup check

Confirm the session has what we'll need: the managed-LLM env vars (`OPENAI_API_BASE` / `OPENAI_API_KEY`) and `kubectl` wired to the `nrp-training-k8s` namespace. No GPU required.

In [ ]:
import os, shutil, subprocess

print("OPENAI_API_BASE =", os.environ.get("OPENAI_API_BASE", "NOT SET"))
# Print the actual token. This is a shared workshop key — fine to show here;
# treat your own personal token as a secret.
key = os.environ.get("OPENAI_API_KEY", "")
print("OPENAI_API_KEY  =", key if key else "NOT SET")

kubectl = shutil.which("kubectl") or "/opt/conda/bin/kubectl"
print("kubectl path    =", kubectl)
print("kubectl version =", subprocess.run([kubectl, "version", "--client", "-o", "json"],
                                          capture_output=True, text=True).stdout[:120], "...")

print("can list pods in nrp-training-k8s:",
      subprocess.run([kubectl, "auth", "can-i", "list", "pods", "-n", "nrp-training-k8s"],
                     capture_output=True, text=True).stdout.strip())


## 2. Use NRP's managed LLMs

NRP runs a **catalog of open-weights LLMs** for you behind one
OpenAI-compatible URL — you send HTTP requests, run no pod, and no GPU is billed
to you. You reach it two ways: a **browser chat UI** at
[`https://nrp-openwebui.nrp-nautilus.io`](https://nrp-openwebui.nrp-nautilus.io),
and the **OpenAI-compatible API** at `https://ellm.nrp-nautilus.io/v1` (used
here). You authenticate with a **bearer token** — mint one at
[`https://nrp.ai/llmtoken`](https://nrp.ai/llmtoken); on this session it's
already exported as `OPENAI_API_KEY`.

**The catalog rotates** as the open-source frontier moves. Roughly, today:

| Model | Size | Good for |
|---|---|---|
| `qwen3` | 397B | frontier reasoning, largest context |
| `gpt-oss` | 120B | agentic tool-calling / code |
| `gemma` | 31B | general chat, multimodal |
| `kimi`, `glm-5`, `minimax-m2` | 230B–1T | coding (under evaluation) |
| `qwen3-embedding` | 8B | embeddings only (used for RAG below) |

> Some of these (`qwen3`, `minimax-m2`, `gpt-oss`, …) are **reasoning models**:
> they write a private chain-of-thought into a separate `reasoning` field before
> filling `content`, so they need a **larger `max_tokens`** or they hit the
> limit before answering. The `chat()` helper below handles that.

Because it's just the OpenAI API, the *same* Python works against the managed
endpoint, a local server, or the OpenAI cloud — **only `base_url` changes**.

**NRP LLM docs:**
[overview](https://nrp.ai/documentation/userdocs/ai/llm-managed/) ·
[models](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) ·
[API access](https://nrp.ai/documentation/userdocs/ai/llm-managed/api-access/) ·
[client configs](https://nrp.ai/documentation/userdocs/ai/llm-managed/client-configs/)

In [ ]:
# List the models currently served by the NRP managed endpoint.
import os, json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ["OPENAI_API_BASE"])
models = client.models.list()
print(f"{len(models.data)} models available right now:\n")
for m in sorted(models.data, key=lambda x: x.id):
    print(f"  {m.id}")


In [ ]:
# A tiny helper we'll reuse across this notebook. The default model is
# `minimax-m2` (fast on NRP, and a good default). It's a *reasoning* model: it
# streams a private chain-of-thought into a separate `reasoning` field and only
# fills `content` once it concludes — so we default `max_tokens` high enough
# (1200) to leave room for both the thinking and the final answer. If a call
# still runs out of tokens mid-thought, we surface that reasoning rather than
# returning nothing.
def chat(prompt, model="minimax-m2", system=None, max_tokens=1200, llm=None):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    resp = (llm or client).chat.completions.create(
        model=model, messages=msgs, max_tokens=max_tokens, temperature=0.2,
    )
    msg = resp.choices[0].message
    if msg.content:
        return msg.content
    reasoning = getattr(msg, "reasoning", None) or getattr(msg, "reasoning_content", None)
    if reasoning:
        return f"(no final answer within max_tokens={max_tokens}; reasoning so far:)\n{reasoning}"
    return "(model produced no content — try raising max_tokens or another model)"

In [ ]:
# Single chat completion.
print(chat(
    "What is the National Research Platform?",
    system="Answer in one sentence.",
))


In [ ]:
# Switch to a smaller model — same code, just change the `model` arg.
print(chat("Explain Kubernetes namespaces in two sentences.", model="gemma-small"))

# Now a reasoning model. We give it room (max_tokens=800) so it finishes
# thinking AND produces a final answer — with too few tokens the whole budget
# is spent reasoning and `content` comes back empty (what the helper guards).
print("\n--- minimax-m2 (a reasoning model) ---")
print(chat("In one sentence, name a strength of this model.", model="minimax-m2", max_tokens=800))

# Try the others yourself — e.g. model="gpt-oss" (fast) or model="qwen3"
# (the 397B flagship; highest quality but slowest to respond).


## 3. RAG — answer from NRP's own docs (no extra infrastructure)

**RAG** ("retrieval-augmented generation") makes an LLM answer from *your*
text instead of guessing: (1) split your docs into chunks and **embed** each
into a vector, (2) embed the question and find the closest chunks, (3) put
those chunks in the prompt with "answer only from this context."

The nice part on NRP: the **embedding model is managed too**. The same endpoint
that serves the chat models also serves `qwen3-embedding`, so we don't install
or download anything. And for a small corpus, a handful of vectors in NumPy is
all the "vector database" we need.

We'll pull **one real page of the NRP documentation** and answer questions from it.


In [ ]:
# Pull one real NRP documentation page (the cluster policies page) and clean it.
import requests, re

PAGE = "https://nrp.ai/documentation/userdocs/start/policies/"
RAW  = ("https://gitlab.nrp-nautilus.io/prp/nrp-site/-/raw/main/"
        "src/content/docs/Documentation/userdocs/start/policies.mdx")

md = requests.get(RAW, timeout=30).text
md = re.sub(r"^---.*?---\s*", "", md, flags=re.S)        # drop frontmatter
md = re.sub(r"^import .*$", "", md, flags=re.M)          # drop MDX imports
md = re.sub(r":::\w+(\[[^\]]*\])?|:::", "", md)          # drop admonition markers
md = re.sub(r"<[^>]+>", "", md)                          # drop stray HTML/JSX
md = re.sub(r"\n{3,}", "\n\n", md).strip()

# Split into ~700-char chunks. (A real corpus would chunk every page; one is plenty here.)
chunks = [md[i:i+700] for i in range(0, len(md), 550)]
print(f"Pulled {len(md)} chars from {PAGE}")
print(f"Split into {len(chunks)} chunks.")


In [ ]:
# Embed with NRP's MANAGED embedding model — no local model, nothing to download.
# Same `client`, same endpoint as the chat models; just a different call.
import numpy as np

def embed(texts):
    resp = client.embeddings.create(model="qwen3-embedding", input=texts)
    vecs = np.array([d.embedding for d in resp.data])
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)   # normalize for cosine

chunk_vecs = embed(chunks)          # one API call embeds the whole page
print(f"Embedded {len(chunks)} chunks into vectors of dim {chunk_vecs.shape[1]} (via qwen3-embedding).")

def retrieve(question, k=3):
    qv = embed([question])[0]
    scores = chunk_vecs @ qv         # cosine similarity (everything is normalized)
    top = scores.argsort()[-k:][::-1]
    return [(chunks[i], float(scores[i])) for i in top]


In [ ]:
# Answer a question by RAG, using NRP's managed LLM.
SYSTEM = ("Answer the question using ONLY the provided context. "
          "If the context doesn't contain the answer, say so. Be concise.")

def ask(question, model="minimax-m2"):
    context = "\n\n".join(text for text, _ in retrieve(question))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=SYSTEM, model=model)

QUESTION = "Is it okay to run `sleep infinity` in a Job on NRP?"
print("Retrieved chunks:")
for text, score in retrieve(QUESTION):
    print(f"  score={score:.3f}  {text[:70].strip()}...")

print("\n--- Answer (NRP managed minimax-m2) ---")
print(ask(QUESTION))

**What to notice.**

- **Everything came from NRP's managed endpoint** — the chat model *and* the
  `qwen3-embedding` embedding model. No models installed or downloaded, and no
  GPU needed.
- Retrieval was ~5 lines of NumPy. That's all a vector database does under the
  hood; for one page it's overkill. For a **large, persistent corpus** you'd
  swap the NumPy array for NRP's managed **Milvus** vector database — same three
  steps (embed, search, prompt), just scalable.
- You can point this exact pattern at **any** docs — in Part 3 we reuse it to
  query the JupyterHub documentation while customizing a hub.

## Takeaways

- The same OpenAI-compatible Python code talked to NRP's managed LLM endpoint —
  list models, chat, swap models — just by pointing `base_url` at
  `ellm.nrp-nautilus.io`. No GPU, no servers to run.
- Managed LLMs are the lowest-friction path for classrooms — no token handoff,
  no GPU billed to students.
- RAG is "embed your docs, retrieve the closest chunks, prompt the LLM with
  'answer from context only.'" Here both the embedding model
  (`qwen3-embedding`) and the LLM came from NRP's managed endpoint, and a few
  NumPy lines replaced a vector database.

**Next:** [`3_jupyterhub.ipynb`](3_jupyterhub.ipynb) — deploy your *own*
JupyterHub on NRP with Helm, and reuse this RAG pattern to navigate the
JupyterHub docs.